In [22]:
import pickle
import matminer
import numpy as np
import pandas as pd
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from matminer.featurizers.composition import ElementProperty, ElementFraction
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, f1_score, precision_score, recall_score

In [23]:
# Load models
with open('sc_ep_best_model_yet.pkl', 'rb') as file:
    model_ep = pickle.load(file)
with open('sc_efep_best_model_yet.pkl', 'rb') as file:
    model_efep = pickle.load(file)
with open('sc_ef_best_model_yet.pkl', 'rb') as file:
    model_ef = pickle.load(file)

In [24]:
# Split the dataframe into features and target
def create_composition(formula):
    try:
        return Composition(formula)
    except ValueError:
        print(f"Error parsing formula: {formula}")

def featurize(data):
    data = data.dropna()
    ep_featurizer = ElementProperty.from_preset('magpie')
    ep_ftd = ep_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    
    # Use ElementFraction instead of ElementProperty
    ef_featurizer = ElementFraction()
    ef_ftd = ef_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
    return ep_ftd, ef_ftd

In [25]:
data = load_dataset("superconductivity2018")
data = data[data["Tc"] >= 10]
print(len(data))
data = data['composition'].to_frame()
#.iloc[0:700]

ef_ftd = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna().reset_index()
ep_ftd = pd.read_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv").dropna().reset_index()

data = data.drop_duplicates()
ef_ftd = ef_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])
ep_ftd = ep_ftd.drop_duplicates(subset=["Critical Temp", "_Composition"])

ef_ftd = ef_ftd[ef_ftd["Critical Temp"] >= 10]
ep_ftd = ep_ftd[ep_ftd["Critical Temp"] >= 10]

6260


In [26]:
ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,6,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,8.0,57.0,49.0,24.195714,18.509388,8.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,10,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,41.0,81.0,40.0,44.412500,5.118750,41.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,11,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,8.0,82.0,74.0,24.310044,18.375508,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,16392,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,8.0,66.0,58.0,24.772201,18.002594,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16393,16393,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,8.0,83.0,75.0,24.242661,19.035620,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16400,16400,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,8.0,57.0,49.0,24.403214,18.746531,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [27]:
data.shape

(6260, 1)

In [28]:
ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,6,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,8.0,57.0,49.0,24.195714,18.509388,8.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,10,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,41.0,81.0,40.0,44.412500,5.118750,41.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,11,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,8.0,82.0,74.0,24.310044,18.375508,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,16392,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,8.0,66.0,58.0,24.772201,18.002594,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16393,16393,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,8.0,83.0,75.0,24.242661,19.035620,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16400,16400,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,8.0,57.0,49.0,24.403214,18.746531,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16403,16403,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [32]:
def record_indices(ep_ftd, ef_ftd, data):
    # Check if composition exists in the original dataset
    data["_Composition"] = data['composition'].apply(create_composition).to_frame()
    compositions = data["composition"].tolist()
    print("Composition:", len(compositions))
    present_indices = []
    missing_indices = []
    
    for idx, composition in enumerate(compositions):
        # Check if the composition exists in the 'composition' column of ep_ftd DataFrame
        if composition in ep_ftd['composition'].tolist():
            # Find the index of the composition in ep_ftd DataFrame
            found_index = ep_ftd[ep_ftd['composition'] == composition]["index"].tolist()
            # Save the found index (or indices) in the present_indices list
            present_indices.extend(found_index)
        else:
            # If the composition is not found, save the enumeration index in the missing_indices list
            missing_indices.append(idx)
    
    print("PRESENT INDICES", present_indices[-10:])
    print("MISSING INDICES", missing_indices)

    # Initialize variables to hold featurized data
    present_ep_ftd = pd.DataFrame()
    present_ef_ftd = pd.DataFrame()
    missing_ep_ftd = pd.DataFrame()
    missing_ef_ftd = pd.DataFrame()

    # If compositions are present, locate them in the featurized data
    if present_indices:
        present_ep_ftd = ep_ftd.loc[ep_ftd["index"].isin(present_indices)]
        present_ef_ftd = ef_ftd.loc[ef_ftd["index"].isin(present_indices)]
    
    # Featurize missing compositions
    if missing_indices:
        missing_data = data.iloc[missing_indices]
        missing_ep_ftd, missing_ef_ftd = featurize(missing_data)

    # Combine present and missing featurized data
    final_ep_ftd = pd.concat([present_ep_ftd, missing_ep_ftd], ignore_index=True)
    final_ef_ftd = pd.concat([present_ef_ftd, missing_ef_ftd], ignore_index=True)

    final_nn_ep_ftd = final_ep_ftd.dropna()
    final_nn_ef_ftd = final_ef_ftd.dropna()
    
    final_in_ep_ftd = final_ep_ftd[final_ep_ftd.isnull().any(axis = 1)]["composition"].tolist()
    final_in_ef_ftd = final_ef_ftd[final_ef_ftd.isnull().any(axis = 1)]["composition"].tolist()
    print("Got Here")
    print(final_in_ep_ftd,"\n", final_in_ef_ftd)
    
    return final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd

final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd = record_indices(ep_ftd, ef_ftd, data)

Error parsing formula: Eu1.45Pr0.05Ce0.5Sr2Cu2Nb1O10=z
Error parsing formula: Sm1Ba-1Cu3O6.94
Error parsing formula: Y2C2Br0.5!1.5
Error parsing formula: Hg0.3Pb0.7Sr1.75La0.25Cu1O4+2
Error parsing formula: Hg1Sr2Ho0.333Ce0.667Cu2O6=z
Error parsing formula: B1Sr2Ca3Cu4O2N+3
Error parsing formula: B1Sr2Ca4Cu5O2N+3
Error parsing formula: B1Sr2Ca2Cu3O2N+3
Composition: 6260
PRESENT INDICES [16378, 16381, 16387, 16389, 16390, 16392, 16393, 16400, 16403, 16404]
MISSING INDICES [765, 1718, 3685, 3700, 4376, 5124, 5296, 5492, 6232]


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementFraction:   0%|          | 0/1 [00:00<?, ?it/s]

Got Here
['Tl0.5Pb0.5Sr4Cu2C1O10'] 
 ['Tl0.5Pb0.5Sr4Cu2C1O10']


In [33]:
final_nn_ep_ftd

,index,composition,Critical Temp,_Composition,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0.0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,19.0,56.0,37.0,30.360000,6.214400,26.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,1.0,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,8.0,57.0,49.0,22.677795,16.074864,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
2,6.0,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,8.0,57.0,49.0,24.195714,18.509388,8.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
3,10.0,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,41.0,81.0,40.0,44.412500,5.118750,41.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
4,11.0,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,8.0,82.0,74.0,24.310044,18.375508,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6246,16392.0,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,8.0,66.0,58.0,24.772201,18.002594,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
6247,16393.0,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,8.0,83.0,75.0,24.242661,19.035620,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
6248,16400.0,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,8.0,57.0,49.0,24.403214,18.746531,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
6249,16403.0,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,8.0,82.0,74.0,27.138250,19.616202,8.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [34]:
def preprocess_for_model1(ep_ftd):
    # Preprocessing steps for model1
    ep_X = ep_ftd.iloc[:, 4:]
    ep_y = ep_ftd['Critical Temp']
    return ep_X, ep_y

# def preprocess_for_model3(ef_ftd):
#     # Preprocessing steps for model3
#     ef_X = ef_ftd.iloc[:, 4:]
#     ef_y = ef_ftd['Critical Temp']
#     return ef_X, ef_y

# def preprocess_for_model2(ep_ftd, ef_ftd):
#     ef_ftd = ef_ftd.iloc[:, 2:]
#     ep_ftd = ep_ftd.iloc[:, 2:]
    
#     print(ef_ftd.shape)
#     print(ep_ftd.shape)
    
#     merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
#     merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")

#     efep_X = merged_df.iloc[:, 2:]
#     efep_y = merged_df['Critical Temp']    
    
#     print(efep_X.shape)
#     return efep_X, efep_y, merged_df

In [10]:
ep_X, ep_y = preprocess_for_model1(final_nn_ep_ftd)
# efep_X, efep_y, merged_df = preprocess_for_model2(ep_ftd, ef_ftd)
# ef_X, ef_y = preprocess_for_model3(ef_ftd)

pred_ep = model_ep.predict(ep_X)
# pred_efep = model_efep.predict(efep_X)
# pred_ef = model_ef.predict(ef_X)

In [11]:
ep_X

,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,19.0,56.0,37.0,30.360000,6.214400,26.0,3.0,84.0,81.0,56.680000,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,8.0,57.0,49.0,22.677795,16.074864,8.0,7.0,87.0,80.0,64.406832,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,8.0,57.0,49.0,24.195714,18.509388,8.0,8.0,87.0,79.0,62.312857,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,41.0,81.0,40.0,44.412500,5.118750,41.0,47.0,80.0,33.0,55.100000,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,8.0,82.0,74.0,24.310044,18.375508,8.0,8.0,87.0,79.0,66.257642,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16392,8.0,66.0,58.0,24.772201,18.002594,8.0,9.0,87.0,78.0,65.378378,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16393,8.0,83.0,75.0,24.242661,19.035620,8.0,7.0,87.0,80.0,68.735160,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16400,8.0,57.0,49.0,24.403214,18.746531,8.0,8.0,87.0,79.0,62.416071,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16403,8.0,82.0,74.0,27.138250,19.616202,8.0,7.0,87.0,80.0,67.885125,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [12]:
print(ep_y)

0        31.20
1        40.10
6        33.00
10       18.20
11       28.10
         ...  
16392    13.00
16393    22.00
16400    19.25
16403    63.60
16404    34.80
Name: Critical Temp, Length: 6251, dtype: float64


In [38]:
# Assuming 'y_true' contains the true labels and 'y_pred' contains the predictions from your ensemble
MSE = mean_squared_error(ep_y, pred_ep)
print("Mean Squared Error:", MSE)

# Confusion Matrix
R2_Score = r2_score(ep_y, pred_ep)
print("R^2 Score:", R2_Score)

# Precision
MAE = mean_absolute_error(ep_y, pred_ep)
print("Mean Absolute Error:", MAE)

# Recall
f1 = f1_score(ep_y, pred_ep)
print("F1 Score", f1)

# F1 Score
precision = precision_score(ep_y, pred_ep)
print("F1 Score:", precision)

# ROC-AUC Score
recall = recall_score(ep_y, pred_ep)
print("ROC-AUC Score:", recall)

Mean Squared Error: 41.26203580761569
R^2 Score: 0.9519656650650441
Mean Absolute Error: 3.6186854221456874


ValueError: continuous is not supported

In [ ]:
# Accuracy: 0.9748884951426651
# Confusion Matrix:
#  [[9836  284]
#  [ 127 6120]]
# Precision: 0.9556527170518426
# Recall: 0.9796702417160237
# F1 Score: 0.9675124496087266
# ROC-AUC Score: 0.9758035003046521

In [ ]:
# Accuracy: 0.9747343092039302
# Confusion Matrix:
#  [[6026  178]
#  [  74 3696]]
# Precision: 0.9540526587506454
# Recall: 0.9803713527851459
# F1 Score: 0.967032967032967
# ROC-AUC Score: 0.9758400928980533